# AI Crop Disease Detector — full training pipeline

Run this notebook on **Kaggle** (Settings -> Accelerator -> GPU T4/P100) or **Google Colab** (Runtime -> GPU).

It reproduces the whole `ml/` pipeline in order:

1. clone the repo (or upload it as a Kaggle Dataset / attach via git)
2. install deps (`ml/requirements.txt`)
3. download the three datasets (needs Kaggle API token — only once)
4. `prepare_data.py` — merge + stratified split (seed 42)
5. `train.py` — two-stage transfer learning (the long cell: 1-2 h on a T4)
6. `evaluate.py` + `calibrate_threshold.py` + `export_onnx.py`
7. download the artifacts: `metrics.md`, `threshold.json`, `labels.json`, `model.onnx`

**IMPORTANT — honesty list:** all numbers this notebook produces are real;
copy them into `ml/reports/` afterwards. Never invent/find-tune past results.

## 1. Setup: clone repo + install dependencies

In [ ]:
# 1a. Get the repository into the machine.
# Kaggle: attach this notebook to a 'Dataset' that IS the repo, or clone:
!git clone https://github.com/ibrahimSumon47/AI-Crop-Disease-Detector-for-Bangladeshi-Farmers.git
%cd AI-Crop-Disease-Detector-for-Bangladeshi-Farmers
!git pull  # make sure we have the latest commits before training

In [ ]:
# 1b. Install pinned ML deps. Kaggle ships a compatible TensorFlow already;
# this makes Colab/Kaggle identical to local runs.
!pip install -q -r ml/requirements.txt
import tensorflow as tf
print('TF version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

## 2. Download the three datasets (only once, needs Kaggle API token)

In [ ]:
# Kaggle API token — no secrets pasted here.
    # 1) kaggle.com -> Settings -> API -> Create New API Token
    # 2) upload `kaggle.json` using the folder '+' button in the right panel of Kaggle
    #    (Colab: files.upload() or !cp /content/drive/MyDrive/kaggle.json .kaggle/kaggle.json)
!mkdir -p ~/.kaggle
# Kaggle/Colab 'Dataset' upload path:
    !test -f /kaggle/input/kaggle-json/kaggle.json && cp /kaggle/input/kaggle-json/kaggle.json ~/.kaggle/kaggle.json
# Colab fallback:
    !test -f drive/MyDrive/kaggle.json && cp drive/MyDrive/kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json
!python -c 'import os; os.path(os.path.expanduser("~/.kaggle/kaggle.json"))' 2>/dev/null || echo 'WARNING: kaggle.json not found — paste it manually'

In [ ]:
# 2b. Download the three datasets into ml/data/raw (small band usage; see
# ml/src/download_data.md for manual alternatives).
!mkdir -p ml/data/raw
# PlantVillage (potato + tomato classes we take; download zipped ~2 GB)
!kaggle datasets download -d abdallahalidev/plantvillage-dataset -p ml/data/raw --unzip
# Rice leaf dataset (choose the version you verified in ml/DATASETS.md)
!kaggle datasets download -d shayanriyaz/riceleafs -p ml/data/raw --unzip
# PlantDoc (field photos; the repo also ships a CSV variant)
!git clone -q https://github.com/pratikm141/PlantDoc.git ml/data/raw/plantdoc
!ls ml/data/raw

**Folder-name mapping note:** if the downloaded folder names differ from
what `prepare_data.py`'s `ALIAS_MAP` expects (e.g. `Rice___LeafBlast`),
either update `ALIAS_MAP` or rename the folders with a quick shell loop.
Afterwards, **verify** with the printed class counts before training.

## 3. Data preparation + sanity statistics

In [ ]:
# Merge datasets and stratified-split into data/processed (seed=42).
# If any class has ~0 images the script tells you — FIX THE DATA before training.
!python ml/src/prepare_data.py
!python ml/src/dataset_stats.py  # -> ml/reports/class_distribution.png
from IPython.display import Image, display
display(Image('ml/reports/class_distribution.png'))

## 4. Training (the long cell — 1–2 h on T4 GPU)

In [ ]:
# Two-stage training. Writes:
#   ml/reports/best_model.keras   (best on val_accuracy)
#   ml/reports/history.json       (full merged history — REAL numbers)
!python ml/src/train.py --epochs-head 5 --epochs-ft 15 --batch 32

In [ ]:
# Optional: plot the merged training history for the report.
import json, matplotlib.pyplot as plt
h = json.load(open('ml/reports/history.json'))['history']
fig, ax = plt.subplots(1, 2, figsize=(12,4))
ax[0].plot(h['accuracy'], label='train'); ax[0].plot(h['val_accuracy'], label='val')
ax[0].set_title('accuracy'); ax[0].legend()
ax[1].plot(h['loss'], label='train'); ax[1].plot(h['val_loss'], label='val')
ax[1].set_title('loss'); ax[1].legend()
plt.savefig('ml/reports/history_curves.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Evaluation, threshold calibration, ONNX export

In [ ]:
# Per-class precision/recall/F1 + confusion matrix -> ml/reports/metrics.md
!python ml/src/evaluate.py --model ml/reports/best_model.keras --data data/processed/test
# Calibrate the confidence threshold on validation data -> threshold.json
!python ml/src/calibrate_threshold.py --model ml/reports/best_model.keras --val data/processed/val
!cat ml/reports/threshold.json

In [ ]:
# Export to ONNX + numerical parity check -> backend/app/data/model.onnx ( + labels.json)
!python ml/src/export_onnx.py --model ml/reports/best_model.keras
!ls -la backend/app/data/

## 6. Collect the artifacts to send back

In [ ]:
# Zip small artifacts for download back into the repo (onnx may be >50MB ->
# then we do NOT commit it; only the metrics/threshold/labels land in git).
%%bash
cd ml/reports
zip -q artifacts.zip metrics.md metrics.py confusion_matrix.png history.json history_curves.png threshold.json train_log_head.csv train_log_finetune.csv ../classes
cp ../../backend/app/data/labels.json .
ls -la artifacts.zip